In [781]:
import sys
import os

sys.path.append(os.path.abspath("../")) 

import torch
from torch import nn
from torch.utils.data import DataLoader
from blocks import CatDogDataset
from utils import get_data_loaders, intersection_over_union
import json
import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import auc

This was the hardest stage of YOLO implementation for me ....

In [782]:
train_loader, val_loader, test_loader = get_data_loaders()

images, targets = next(iter(train_loader))

preds = torch.randn_like(targets)

Decode predictions:

In [783]:
preds.shape

torch.Size([64, 7, 7, 47])

In [784]:
def run_nms(boxes_for_some_image, iou_threshold):

    if len(boxes_for_some_image) == 0:
        return []

    boxes_for_some_image = torch.tensor(boxes_for_some_image)
    keep_boxes_for_some_image = []

    classes = boxes_for_some_image[:, 0].unique()

    for class_idx in classes:
        mask = boxes_for_some_image[:, 0] == class_idx
        class_boxes = boxes_for_some_image[mask]

        conf_scores = class_boxes[:, 1]

        sort_indices = torch.argsort(conf_scores, descending=True)

        class_boxes = class_boxes[sort_indices]

        while True:
            keep_boxes_for_some_image.append(class_boxes[0])
            class_boxes = class_boxes[1:]
            saved_boxes = []

            for i in range(class_boxes.shape[0]):

                iou = intersection_over_union(keep_boxes_for_some_image[-1][2:], class_boxes[i][2:])

                if iou < iou_threshold:
                    saved_boxes.append(class_boxes[i])

            if len(saved_boxes) == 0:
                break

            class_boxes = torch.stack(saved_boxes)
            
            
    return [box.tolist() for box in keep_boxes_for_some_image]


In [ ]:
def decode_preds(preds, S=7, B=2, conf_score_threshold=0.5, iou_threshold=0.5):
    preds = preds.reshape(preds.shape[0], -1, preds.shape[-1])

    batch_size = preds.shape[0]
    grid_cells_n = preds.shape[1] 

    all_preds_boxes = []

    for i in range(batch_size):
        
        bboxes_for_this_image = []

        for j in range(grid_cells_n):
            
            row = j // S
            col = j % S

            class_probs = preds[i, j, 10:]
            max_class_prob, class_id = torch.max(class_probs, dim=0)
            class_id = class_id.item()

            for b in range(B):
                box_conf = preds[i, j, 5*b + 4].item()
                # Final confidence = Pr(Object) * Pr(Class | Object)
                final_conf = box_conf * max_class_prob.item()

                if final_conf > conf_score_threshold:
                    
                    x_local = preds[i, j, 5*b + 0].item()
                    y_local = preds[i, j, 5*b + 1].item()
                    w = preds[i, j, 5*b + 2].item()
                    h = preds[i, j, 5*b + 3].item()

                    x_global = (x_local + col) / S
                    y_global = (y_local + row) / S

                    clean_box = [class_id, final_conf, x_global, y_global, w, h]
                    
                    bboxes_for_this_image.append(clean_box)

        bboxes_for_this_image = run_nms(bboxes_for_this_image, iou_threshold)

        for box in bboxes_for_this_image:
            final_row = [i] + box 
            all_preds_boxes.append(final_row)

    return all_preds_boxes

In [786]:
preds = decode_preds(preds)

len(preds)

1962

Decode targets:

In [787]:
targets.shape

torch.Size([64, 7, 7, 47])

In [788]:
def decode_targets(targets, S=7, B=2):

    batch_size = targets.shape[0]
    grid_cells_n = S*S

    targets = targets.reshape(batch_size, grid_cells_n, -1)

    all_target_boxes = []

    for i in range(batch_size):

        bboxes_for_this_image = []

        for j in range(grid_cells_n):

            row = j // S
            col = j % S

            class_id = torch.argmax(targets[i, j, 10:]).item()

            conf = targets[i, j, 4].item()

            if conf > 0:
                x_local = targets[i, j, 0].item()
                y_local = targets[i, j, 1].item()
                w = targets[i, j, 2].item()
                h = targets[i, j, 3].item()

                x_global = (x_local + col) / S
                y_global = (y_local + row) / S

                clean_box = [class_id, conf, x_global, y_global, w, h]
                
                bboxes_for_this_image.append(clean_box)
        
        for box in bboxes_for_this_image:
            final_row = [i] + box
            all_target_boxes.append(final_row)

    return all_target_boxes


In [789]:
targets = decode_targets(targets)

len(targets)

64

Now let's build mAP:

In [ ]:
def compute_mAP(preds, targets, iou_threshold, num_classes):
    decoded_preds = decode_preds(preds)
    decoded_targets = decode_targets(targets)
    
    preds = torch.tensor(decoded_preds).view(-1, 7) if len(decoded_preds) > 0 else torch.zeros((0, 7))
    targets = torch.tensor(decoded_targets).view(-1, 7) if len(decoded_targets) > 0 else torch.zeros((0, 7))
    
    average_precisions = []

    for class_idx in range(num_classes):
        
        class_preds = preds[preds[:, 1] == class_idx]
        class_targets = targets[targets[:, 1] == class_idx]

        if len(class_targets) == 0 and len(class_preds) == 0:
            continue
            
        if len(class_preds) == 0:
            average_precisions.append(0.0)
            continue

        sort_idx = torch.argsort(class_preds[:, 2], descending=True)
        class_preds = class_preds[sort_idx]

        metrics_table = torch.zeros((len(class_preds), 2))
        
        targets_matched = {}
        for img_idx in class_targets[:, 0].unique():
            img_targets_count = len(class_targets[class_targets[:, 0] == img_idx])
            targets_matched[img_idx.item()] = torch.zeros(img_targets_count)

        for i, pred in enumerate(class_preds):
            img_idx = pred[0].item()
            
            img_targets = class_targets[class_targets[:, 0] == img_idx]
            
            if len(img_targets) == 0:
                metrics_table[i, 1] = 1 
                continue

            iou_scores = torch.zeros(len(img_targets))
            for j, target in enumerate(img_targets):
                if targets_matched[img_idx][j] == 0:
                    iou_scores[j] = intersection_over_union(pred[3:], target[3:])
                else:
                    iou_scores[j] = -1

            max_iou_idx = torch.argmax(iou_scores)
            max_iou = iou_scores[max_iou_idx]

            if max_iou >= iou_threshold:
                metrics_table[i, 0] = 1 
                targets_matched[img_idx][max_iou_idx.item()] = 1
            else:
                metrics_table[i, 1] = 1 
                
        cum_tp = torch.cumsum(metrics_table[:, 0], dim=0)
        cum_fp = torch.cumsum(metrics_table[:, 1], dim=0)

        total_targets = len(class_targets)
        
        recall = cum_tp / (total_targets + 1e-8)
        precision = cum_tp / (cum_tp + cum_fp + 1e-8)

        precision = torch.cat((torch.tensor([1.0]), precision))
        recall = torch.cat((torch.tensor([0.0]), recall))
        
        precision = torch.flip(torch.cummax(torch.flip(precision, dims=[0]), dim=0)[0], dims=[0])

        AP = torch.trapz(y=precision, x=recall).item()
        
        average_precisions.append(AP)

    if len(average_precisions) == 0:
        return 0.0
        
    mAP = sum(average_precisions) / len(average_precisions)
    return mAP

In [791]:
compute_mAP(preds=preds, targets=targets, iou_threshold=0.1, num_classes=37)

0.010957679912649296